# E8 — Error analysis tại máy cá nhân

Chỉ từ predictions E1 đã xác minh; không inference. Các nhóm pairwise có thể giao nhau; all-six-miss dùng đủ sáu predictors. Không cherry-pick test-score extremes để thay đổi scope hoặc hyperparameters.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/master_session1.py").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.master_session1 import ARCHIVE_NAME, ARCHIVE_SHA256, BUNDLE_HASH, analyze
TABLES = ROOT / "results/tables/master_session1"
FULL_AUDIT = False  # True needs the private original ZIP and matching dataset; never trains.
REBUILD_FIGURES = False
if FULL_AUDIT:
    archive = Path(os.environ.get("SEQREC_MASTER_ARCHIVE", str(ROOT / "artifacts/master_session1" / ARCHIVE_NAME)))
    bundle = Path(os.environ.get("SEQREC_BUNDLE", str(ROOT / "data/processed/beauty" / BUNDLE_HASH)))
    analyze(archive, ROOT, bundle, resamples=10000)
audit = json.loads((TABLES / "artifact_audit.json").read_text())
assert audit["state"] == "VERIFIED" and audit["archive_sha256"] == ARCHIVE_SHA256
assert audit["E1_success"] == 16 and audit["E2_success"] == 6 and audit["E7_gpu_state"] == "DEFERRED"
print("Source SHA256:", ARCHIVE_SHA256)
print("Mode:", "raw ZIP re-audited" if FULL_AUDIT else "versioned audited aggregate tables; not a fresh raw re-audit")


In [ ]:
groups = pd.read_csv(TABLES / "e8_error_groups.csv")
display(groups)
display(groups.groupby("category").agg(mean_fraction=("fraction", "mean"), sd_fraction=("fraction", "std")))
display(pd.read_csv(TABLES / "e8_pair_outcomes.csv"))


## History length và target popularity

Bins được định nghĩa cố định, không fit dựa trên test performance. Target popularity chỉ đếm training interactions; giữ cold targets. Per-user examples được lưu riêng tư tại results/derived, không commit raw histories/predictions. Examples lấy ba user ID nhỏ nhất mỗi nhóm/seed theo quy tắc cố định, không chọn chênh lệch lớn nhất.


In [ ]:
segments = pd.read_csv(TABLES / "e8_segments.csv")
display(segments)
private_cases = ROOT / "results/derived/master_session1" / ARCHIVE_SHA256 / "e8_private_examples.json"
print("Private cases available locally:", private_cases.is_file())
# Do not display or commit identifying raw histories in this public notebook.


In [ ]:
from gen_fig_master_session1 import generate
if REBUILD_FIGURES:
    generate(ROOT)
display(Image(filename=str(ROOT / "results/figures/master_session1/e8_error_groups.png")))
display(Image(filename=str(ROOT / "results/figures/master_session1/e8_history_popularity.png")))


## Báo cáo và trạng thái

Đọc [báo cáo tiếng Việt](../docs/MASTER_SESSION1_ANALYSIS_VI.md). Phạm vi thu gọn do tài nguyên GPU và yêu cầu môn học, không theo mô hình có test score cao. E1 complete, E2 partial, E3 pending/reuse available, E4/E5 deferred; TIGER/CRAMER theory only. Không tuyên bố các phase chưa chạy đã hoàn tất.
